# G??wny notebook analizy

Ten notebook jest g??wnym miejscem pracy. Kom?rka startowa sama dodaje katalog `src` do ?cie?ki Pythona, wi?c nie trzeba instalowa? projektu jako pakietu.


In [ ]:
from pathlib import Path
import sys

# Dzia?a zar?wno przy uruchomieniu z katalogu g??wnego projektu, jak i z katalogu notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("Source dir:", SRC_DIR)


## Importy

Wszystkie funkcje analityczne s? w `src/mgr_mk/`, ale tutaj importujesz je jednym miejscem.


In [ ]:
import pandas as pd

from mgr_mk.data_loader import load_events, load_lineups, load_world_cup_2018_matches, team_name
from mgr_mk.features import build_player_match_features
from mgr_mk.momentum import build_match_momentum, cumulative_momentum, cumulative_xg, goal_events
from mgr_mk.plots import plot_cumulative_xg, plot_match_momentum


## Wyb?r meczu

Domy?lnie ustawiony jest mecz Brazylia - Belgia (`8650`), bo dobrze pokazuje r??nic? mi?dzy wynikiem, momentum i xG. Mo?esz zmieni? `match_id` na dowolny z tabeli `df_matches`.


In [ ]:
df_matches = load_world_cup_2018_matches()

# Zmie? t? warto??, je?li chcesz analizowa? inny mecz.
match_id = 8650

match_info = df_matches.loc[df_matches["match_id"].eq(match_id)].iloc[0]
home_team = team_name(match_info, "home_team")
away_team = team_name(match_info, "away_team")

print(f"Wybrany mecz: {home_team} {match_info['home_score']} - {match_info['away_score']} {away_team}")
df_matches[["match_id", "match_date", "home_team", "away_team", "home_score", "away_score"]].head()


## ?adowanie danych meczu


In [ ]:
df_events = load_events(match_id)
lineups = load_lineups(match_id)

df = df_events.copy()
print("Events:", df.shape)
df[["minute", "second", "team.name", "player.name", "type.name"]].head()


## Cechy zawodnik?w


In [ ]:
player_match_final = build_player_match_features(df, match_id)
player_match_final.head(10)


## Match momentum

S?upki pokazuj? r??nic? momentum w oknach 5-minutowych, a linie pokazuj? skumulowane momentum obu dru?yn. Logika i wagi s? w `src/mgr_mk/momentum.py` oraz `src/mgr_mk/config.py`.


In [ ]:
fig_momentum, match_momentum = plot_match_momentum(df, df_matches, match_id, window=5)
match_momentum.head(10)


## Gole wykryte w danych

Tu sprawdzisz, czy wykres uwzgl?dnia gole ze strza??w oraz samob?je (`Own Goal For`).


In [ ]:
momentum_table, teams, momentum_events = build_match_momentum(df, match_info, window=5)
goals = goal_events(momentum_events)
goals


## Skumulowane momentum jako tabela


In [ ]:
cum_momentum = cumulative_momentum(momentum_events)
cum_momentum.tail(10)


## Skumulowane xG

Ten wykres jest osobno od momentum. Pokazuje czysto narastaj?ce xG ze strza??w.


In [ ]:
fig_xg = plot_cumulative_xg(df, df_matches, match_id, window=5)
xg_timeline = cumulative_xg(df)
xg_timeline.groupby("team.name")["cum_xg"].max().round(3)


## Szybka zmiana meczu

?eby przeanalizowa? inny mecz, zmie? `match_id` w sekcji **Wyb?r meczu** i uruchom notebook ponownie od tej kom?rki w d??.
